# Zutaten-Normalisierung & Gruppierung mit NLP

## Zielsetzung
Erstellung eines Ähnlichkeits-Caches für Zutaten aus TheMealDB, um:
- **Plural/Singular** zusammenzufassen (z.B. "Tomato" ↔ "Tomatoes")
- **Varianten** zu erkennen (z.B. "Cherry Tomatoes" ↔ "Baby Plum Tomatoes")
- **Modifikatoren** zu behandeln (z.B. "Fresh Basil" → "Basil")

---

## Workflow-Übersicht

```
1. Zutaten-Pool laden
    ↓
2. Grammatik-Analyse (spaCy)
    ↓
3. Token-Cache erstellen
    ↓
4. Ähnlichkeits-Vergleiche
    ↓
5. JSON-Export
```

---

## 1. Grammatik-Analyse mit spaCy

### Kernproblem: "Tomato Sauce" vs. "Tomatoes"
- **HEAD**: Grammatikalisches Zentrum (→ `sauce` / `tomato`)
- **MODIFIER**: Erweiterte Information (→ `tomato` / ∅)

### Manuelle Korrekturen
```python
NOISE_WORDS = {"leaves", "leaf", "seed", "nuts", "yolks"}
MANUAL_CORRECTIONS = {"leaves": "leaf", "nuts": "nut", "yolks": "yolk"}
```

### Funktion: `analyze_grammar(text)`
- Lemmatisierung (z.B. "tomatoes" → "tomato")
- Trennung von HEAD und MODIFIERS
- Ignoriert: Adjektive (`amod`), Präpositionen (`prep`)

**Beispiel:**
```
"Fresh Cherry Tomatoes"
→ HEAD: "tomato"
→ MODIFIER: "cherry"
(Ignoriert: "fresh" als Adjektiv)
```

---

## 2. Token-Vektorisierung

### Vektorkombination (4 Methoden)
Jede Zutat wird in 2 Vektoren zerlegt (HEAD + MODIFIER) und kombiniert:

| Methode | Formel | Vorteil |
|---------|--------|---------|
| `weighted` | 70% HEAD + 30% MOD | Betont grammatikalische Hierarchie |
| `concat` | [HEAD \|\| MOD] | Keine Informationsverlust (600D) |
| `hadamard` | HEAD ⊙ MOD | Betont gemeinsame Features |
| `max` | max(HEAD, MOD) | Robusteste Features beider Vektoren |

**Aktuell genutzt:** `METHOD = "max"`

---

## 3. IngredientTokenCache

### Initialisierung
```python
cache = IngredientTokenCache(ALL_INGREDIENTS, method="max")
```

**Workflow:**
1. Für jede Zutat: `analyze_grammar()` → HEAD + MOD
2. Konvertierung zu spaCy-Tokens (`en_core_web_md` für Vektoren)
3. Kombination zu `combine_tokens[ingredient]`

**Vorteil:** Einmalige Berechnung statt 877² Vergleiche!

---

## 4. Ähnlichkeits-Vergleich

### Funktion: `check_similarity_combined()`
- **Cosinus-Ähnlichkeit** zwischen kombinierten Vektoren
- Threshold: `0.85` (anpassbar)
- Output: `[(ingredient, score), ...]` (sortiert nach Score)

**Beispiel:**
```python
cache.check_similarity("Cucumber", threshold=0.85)
# → {"Cucumber": [("Cucumbers", 0.98), ("Zucchini", 0.87), ...]}
```

---

## 5. JSON-Export

### Dateiformat
```json
{
  "Chicken": [
     ["Chicken Breast", 0.95],
     ["Chicken Legs", 0.92],
     ["Chicken Thighs", 0.91]
  ],
  "Tomato": [
     ["Tomatoes", 0.99],
     ["Cherry Tomatoes", 0.89],
     ["Baby Plum Tomatoes", 0.87]
  ]
}
```

### Dateiname
```
ingredient_similarity_cache_DD-MM-YYYY-HH-MM_{METHOD}.json
```

**Generierung für alle Methoden:**
```python
for method in ["weighted", "concat", "hadamard", "max"]:
     write_JSON_similar_ingredients_fast(method)
```

---

## Validierung

### Funktion: `validate_similarity_cache()`
Prüft:
- ✅ Alle 877 Zutaten haben einen Key
- ✅ Keine leeren Listen
- ⚠️ Identifiziert problematische Fälle (z.B. zu hohe Thresholds)

---

## Probleme & Lösungen

| Problem | Ursache | Lösung |
|---------|---------|--------|
| Plural/Singular getrennt | spaCy erkennt nicht immer Lemma | Manuelle `MANUAL_CORRECTIONS` |
| "Leaves" ≠ "Leaf" | Noise-Word-Handling | `NOISE_WORDS` Set |
| "Walnut Oil" → "Oil" | HEAD dominiert | Modifier-Vektor mit 30% gewichtet |
| JSON-Serialisierung scheitert | `numpy.float32` | Konvertierung zu Python-`float()` |

---

## Performance

- **Ohne Cache:** ~385.000 NLP-Vergleiche (877²)
- **Mit Cache:** ~877 Vergleiche (1× Initialisierung + 1× Lookup)
- **Speedup:** ~440×

In [1]:
import os
import sys

# Pfad zum übergeordneten Verzeichnis hinzufügen (damit themealdb_client gefunden wird)
sys.path.insert(0, os.path.abspath('..'))

from themealdb_client import TheMealDBClient
import json
import traceback

# Test für get_all_ingredients() Funktion

# Client initialisieren
client = TheMealDBClient()
def get_all_ingredients():
    try:
        ingredients = client.get_all_ingredients()
        ingredients = [ingredient['strIngredient'] for ingredient in ingredients if 'strIngredient' in ingredient]
        print(f"Anzahl Zutaten gefunden: {len(ingredients)}")
        print("Beispiel-Zutaten:")
        for ing in ingredients[:10]:  # Zeige die ersten 10 Zutaten
            print(f"- {ing}")
        return ingredients
    except Exception as e:
        print("Fehler beim Abrufen der Zutaten:")
        traceback.print_exc()
        return []
ALL_INGREDIENTS = get_all_ingredients()


Anzahl Zutaten gefunden: 877
Beispiel-Zutaten:
- Chicken
- Salmon
- Beef
- Pork
- Avocado
- Apple Cider Vinegar
- Asparagus
- Aubergine
- Baby Plum Tomatoes
- Bacon


In [2]:
import spacy

# Vektoren funktionieren nur mit md oder lg!
try:
    nlp_score   = spacy.load("en_core_web_md")
    nlp_head    = spacy.load("en_core_web_trf")
except:
    print("Bitte lade das Medium-Modell: python -m spacy download en_core_web_md")
    nlp_score = spacy.load("en_core_web_sm") # Fallback (wird aber Warnung werfen)
    
# 1. NOISE (Ignorieren wir komplett)
NOISE_WORDS = {
    "leaves", "leaf", "seed", "seeds", "nuts", "yolks"
}

MANUAL_CORRECTIONS = {"leaves": "leaf", "leave": "leaf", "nuts": "nut", "yolks": "yolk"}
    
def analyze_grammar(text):
        doc = nlp_head(text.lower())
        
        # 1. Den grammatikalischen Kern finden (ROOT) (tomato sauce -> sauce)
        # Das Wort, das von keinem anderen abhängt
        head_token = [t for t in doc if t.head == t][0]
           
        # 2. Modifiers sammeln (erweiterte information)
        modifiers = [] # Nomen, die den Typ bestimmen (Walnut -> Oil)
        
        for child in head_token.children:
            # # 1. Ignoriere Adjektive (amod) wie "chopped", "fresh", "green"
            # if child.dep_ == "amod":
            #     continue
                
            # # 2. Ignoriere Mengenangaben (nummod) wie "2", "two"
            # elif child.dep_ == "nummod":
            #     continue
                
            # 3. Ignoriere Präpositionen (prep) wie "of" in "Cup of Walnuts"
            if child.dep_ == "prep":
                continue

            else:
                # manuelle Korrektur für modifiers
                modifiers.append(MANUAL_CORRECTIONS.get(child.text, child.lemma_))
                
        # sortiere die Modifier alphabetisch für Konsistenz
        modifiers.sort()
        modifiers_combined = " ".join(modifiers)
        
        # 3. Manuelle Korrekturen für Head (z.B. "leaves" -> "leaf")
        found_head = MANUAL_CORRECTIONS.get(head_token.text, head_token.lemma_)
         
        if found_head in NOISE_WORDS:
            found_head = modifiers_combined
            modifiers_combined = ""
        
        return (modifiers_combined, found_head)


d:\GitRepo\PKI_Projekt_Gruppe_B1_4\.venvRezept\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import numpy as np

METHOD = "max"
# --- Schneller IngredientPool-Index für NLP-Vergleiche ---
class IngredientTokenCache:
    def __init__(self, ingredient_pool, method=METHOD):
        """Berechnet einmalig alle NLP-Tokens für schnellere Vergleiche"""
        print(f"Initialisiere Token-Cache für {len(ingredient_pool)} Zutaten...")
        self.token_dict = {}
        self.combine_tokens = {}
        self.method = method
        for idx, ingredient in enumerate(ingredient_pool):
            if idx % 100 == 0:
                print(f"  Fortschritt: {idx}/{len(ingredient_pool)}")
            self.token_dict[ingredient] = self.get_head_and_mod_token(ingredient)
            self.combine_tokens[ingredient] = self._combine_tokens(*self.token_dict[ingredient])
        print("Token-Cache bereit!")
    
    def _combine_tokens(self, head_token, mod_token, weight_head=0.7, weight_mod=0.3):
        """
        Kombiniert zwei Tokens zu einem neuen Vektor.
        
        Args:
            head_token: spaCy Token für HEAD
            mod_token: spaCy Token für MODIFIER
            weight_head: Gewicht für HEAD (nur bei method="weighted")
            weight_mod: Gewicht für MODIFIER (nur bei method="weighted")
            method: Kombinationsmethode
                - "weighted": Gewichteter Durchschnitt (Standard)
                - "concat": Vektoren konkatenieren (doppelte Länge!)
                - "hadamard": Element-weise Multiplikation
                - "max": Element-weises Maximum
        
        Returns:
            numpy.ndarray: Der kombinierte Vektor
        """
        method = self.method
        if mod_token.text == "":
            return head_token.vector
        
        if method == "weighted":
            # Gewichteter Durchschnitt (70% HEAD + 30% INFO)
            combined_vector = (head_token.vector * weight_head + mod_token.vector * weight_mod) / (weight_head + weight_mod)
            
        if method == "weighted7030":
            # Gewichteter Durchschnitt (70% HEAD + 30% INFO)
            weight_head = 0.7
            weight_mod = 0.3
            combined_vector = (head_token.vector * weight_head + mod_token.vector * weight_mod) / (weight_head + weight_mod)
            
        if method == "weighted5050":
            # Gewichteter Durchschnitt (50% HEAD + 50% INFO)
            weight_head = 0.5
            weight_mod = 0.5
            combined_vector = (head_token.vector * weight_head + mod_token.vector * weight_mod) / (weight_head + weight_mod)
        
        elif method == "concat":
            # Einfach beide Vektoren aneinanderhängen
            # Vorteil: Keine Information geht verloren
            # Nachteil: Doppelte Vektorlänge (600D statt 300D)
            combined_vector = np.concatenate([head_token.vector, mod_token.vector])
        
        elif method == "hadamard":
            # Element-weise Multiplikation (Hadamard-Produkt)
            # Vorteil: Betont gemeinsame Features
            # Nachteil: Kann Nullen erzeugen
            combined_vector = head_token.vector * mod_token.vector
        
        elif method == "max":
            # Element-weises Maximum
            # Vorteil: Behält stärkste Features beider Vektoren
            combined_vector = np.maximum(head_token.vector, mod_token.vector)
        
        else:
            raise ValueError(f"Unbekannte Methode: {method}. Verwende 'weighted', 'concat', 'hadamard' oder 'max'")
        
        return combined_vector
    
    
    def get_head_and_mod_token(self, ingredient):
        """Konvertiert eine Zutat in NLP-Tokens (Head + Modifiers)"""
        modifiers_combined, ingredient_head = analyze_grammar(ingredient)
        return nlp_score(ingredient_head), nlp_score(modifiers_combined)
    
    def get_head_and_mod_token_combined(self, ingredient):
        """Konvertiert eine Zutat in NLP-Tokens (Head + Modifiers)"""
        modifiers_combined, ingredient_head = analyze_grammar(ingredient)
        # kombiniere token zu einem neuen Vektor mit Gewichtung
        combined_token = self._combine_tokens(nlp_score(ingredient_head), nlp_score(modifiers_combined))
        
        return combined_token
    
    
    def check_similarity(self, baseIngredient, thresholdHead=0.8, thresholdMod=0.8, debug=False):
        """
        Vergleicht baseIngredient mit allen gecachten Zutaten.
        Returns: { baseIngredient: [(ingredient, scoreHead, scoreMod), ...] }
        """
        baseTokenHead, baseTokenModifiersCombined = self.get_head_and_mod_token(baseIngredient)
        results = []
        
        for comparableIngredient, (comparableTokenHead, comparableTokenModifiersCombined) in self.token_dict.items():
            scoreHead = baseTokenHead.similarity(comparableTokenHead)
            scoreModifiersCombined = -10
            
            if baseTokenModifiersCombined.text and comparableTokenModifiersCombined.text:
                scoreModifiersCombined = baseTokenModifiersCombined.similarity(comparableTokenModifiersCombined)
            
            if debug:
                print(f"{comparableIngredient:<30} | Head: {scoreHead:.4f} | Mod: {scoreModifiersCombined:.4f}")
            
            if (scoreHead > thresholdHead and not scoreHead > 1.0) and (scoreModifiersCombined > thresholdMod or scoreModifiersCombined == -10):
                results.append((comparableIngredient, float(scoreHead), float(scoreModifiersCombined)))
        
        results.sort(key=lambda x: x[1], reverse=True)
        
        if debug:
            for comparableIngredient, scoreHead, scoreModifiersCombined in results:
                barHead = "█" * int(scoreHead * 10)
                barModifiersCombined = "█" * int(scoreModifiersCombined * 10)
                print(f"{comparableIngredient:<20} | {scoreHead:.4f} {barHead}  {scoreModifiersCombined:.4f} {barModifiersCombined}")
        
        return {baseIngredient: results}
    
    def check_similarity_combined(self, baseIngredient, threshold=0.8, debug=False):
        """
        Vergleicht baseIngredient mit allen gecachten Zutaten basierend auf dem kombinierten Token.
        Returns: { baseIngredient: [(ingredient, combined_score), ...] }
        """
        baseTokenCombined = self.get_head_and_mod_token_combined(baseIngredient)
        results = []
        
        for comparableIngredient in self.combine_tokens.keys():
            comparableTokenCombined = self.combine_tokens[comparableIngredient]
            
            # Cosinus-Ähnlichkeit für NumPy-Arrays
            combined_score = np.dot(baseTokenCombined, comparableTokenCombined) / (np.linalg.norm(baseTokenCombined) * np.linalg.norm(comparableTokenCombined))
            
            if debug:
                print(f"{comparableIngredient:<30} | Combined Score: {combined_score:.4f}")
            
            if combined_score > threshold and combined_score <= 1.0:
                # FIX: Konvertiere numpy.float32 zu Python-float für JSON-Serialisierung
                results.append((comparableIngredient, float(combined_score)))
        
        results.sort(key=lambda x: x[1], reverse=True)
        
        if debug:
            for comparableIngredient, combined_score in results:
                barCombined = "█" * int(combined_score * 10)
                print(f"{comparableIngredient:<20} | {combined_score:.4f} {barCombined}")
        
        return {baseIngredient: results}
        

# Beispielnutzung:
# cache = IngredientTokenCache(ALL_INGREDIENTS)
# result = cache.check_similarity("Tomato", thresholdHead=0.9)


In [4]:
import json
from datetime import datetime

def write_JSON_similar_ingredients_fast(method=METHOD):
    """
    Erstellt eine JSON-Datei mit ähnlichen Zutaten für alle Zutaten.
    Nutzt IngredientTokenCache für schnellere Verarbeitung.
    Format: { "Zutat": ["Ähnliche1", "Ähnliche2", ...] }
    Dateiname: ingredient_similarity_cache_DD-MM-YYYY-HH-MM.json
    """
    
    # Cache einmalig initialisieren (das dauert, spart aber enorm Zeit danach)
    cache = IngredientTokenCache(ALL_INGREDIENTS, method=method)
    
    similarity_data = {}
    
    print(f"\nVerarbeite {len(ALL_INGREDIENTS)} Zutaten mit Methode '{method}'...")
    
    for index, ingredient in enumerate(ALL_INGREDIENTS):
        if index % 50 == 0:
            print(f"Fortschritt: {index}/{len(ALL_INGREDIENTS)}")
        
        # Ähnliche Zutaten finden (Head + Modifiers)
        if False:
            result = cache.check_similarity(ingredient, thresholdHead=0.9, thresholdMod=0.8, debug=False)
        
        # Ähnliche Zutaten finden (kombinierter Token)    
        else:
            result = cache.check_similarity_combined(ingredient, threshold=0.85, debug=False)
        
        # Nur die Zutaten-Namen extrahieren
        if ingredient in result and result[ingredient]:
            similarity_data[ingredient] = list(result[ingredient])
    
    # Dateiname mit Datum erstellen
    timestamp = datetime.now().strftime("%d-%m-%Y-%H-%M")
    output_file = f"ingredient_similarity_cache_{timestamp}_{method}.json"
    
    # JSON speichern (nur das Dictionary)
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(similarity_data, f, indent=2, ensure_ascii=False)
    
    print(f"\nFertig! Datei gespeichert: {output_file}")
    print(f"Zutaten mit Matches: {len(similarity_data)}")
    
    return similarity_data

# Funktion ausführen (jetzt mit Cache - viel schneller!)
for method in ["weighted", "concat", "hadamard", "max"]:
    similarity_results = write_JSON_similar_ingredients_fast(method)

Initialisiere Token-Cache für 877 Zutaten...
  Fortschritt: 0/877


d:\GitRepo\PKI_Projekt_Gruppe_B1_4\.venvRezept\lib\site-packages\thinc\shims\pytorch.py:114: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(self._mixed_precision):


  Fortschritt: 100/877
  Fortschritt: 200/877
  Fortschritt: 300/877
  Fortschritt: 400/877
  Fortschritt: 500/877
  Fortschritt: 600/877
  Fortschritt: 700/877
  Fortschritt: 800/877
Token-Cache bereit!

Verarbeite 877 Zutaten mit Methode 'weighted'...
Fortschritt: 0/877


C:\Users\maxi9\AppData\Local\Temp\ipykernel_22760\1308947194.py:125: RuntimeWarning: invalid value encountered in scalar divide
  combined_score = np.dot(baseTokenCombined, comparableTokenCombined) / (np.linalg.norm(baseTokenCombined) * np.linalg.norm(comparableTokenCombined))


Fortschritt: 50/877
Fortschritt: 100/877
Fortschritt: 150/877
Fortschritt: 200/877
Fortschritt: 250/877
Fortschritt: 300/877
Fortschritt: 350/877
Fortschritt: 400/877
Fortschritt: 450/877
Fortschritt: 500/877
Fortschritt: 550/877
Fortschritt: 600/877
Fortschritt: 650/877
Fortschritt: 700/877
Fortschritt: 750/877
Fortschritt: 800/877
Fortschritt: 850/877

Fertig! Datei gespeichert: ingredient_similarity_cache_30-01-2026-22-23_weighted.json
Zutaten mit Matches: 828
Initialisiere Token-Cache für 877 Zutaten...
  Fortschritt: 0/877
  Fortschritt: 100/877
  Fortschritt: 200/877
  Fortschritt: 300/877
  Fortschritt: 400/877
  Fortschritt: 500/877
  Fortschritt: 600/877
  Fortschritt: 700/877
  Fortschritt: 800/877
Token-Cache bereit!

Verarbeite 877 Zutaten mit Methode 'concat'...
Fortschritt: 0/877


ValueError: shapes (300,) and (600,) not aligned: 300 (dim 0) != 600 (dim 0)

In [ ]:
def validate_similarity_cache(json_file, all_ingredients_list):
    """
    Validiert die erstellte JSON-Datei:
    - Prüft, ob jede Zutat einen Key hat
    - Prüft, ob die Listen nicht leer sind
    """
    try:
        with open(json_file, "r", encoding="utf-8") as f:
            cache_data = json.load(f)
    except Exception as e:
        print(f"Fehler beim Lesen der Datei: {e}")
        return False
    
    print(f"Validiere {json_file}...")
    print(f"Zutaten in ALL_INGREDIENTS: {len(all_ingredients_list)}")
    print(f"Keys in JSON: {len(cache_data)}")
    
    missing_keys = []
    empty_lists = []
    
    # Prüfe, ob alle Zutaten einen Key haben
    for ingredient in all_ingredients_list:
        if ingredient not in cache_data:
            missing_keys.append(ingredient)
        elif not cache_data[ingredient]:  # Liste ist leer
            empty_lists.append(ingredient)
    
    # Ausgabe der Ergebnisse
    print("\n--- VALIDIERUNGSERGEBNISSE ---")
    
    if missing_keys:
        print(f"\n❌ FEHLER: {len(missing_keys)} Zutaten ohne Key:")
        for ing in missing_keys[:10]:  # Zeige nur die ersten 10
            print(f"  - {ing}")
        if len(missing_keys) > 10:
            print(f"  ... und {len(missing_keys) - 10} mehr")
    else:
        print("✅ Alle Zutaten haben einen Key")
    
    if empty_lists:
        print(f"\n⚠️ WARNUNG: {len(empty_lists)} Zutaten haben leere Listen:")
        for ing in empty_lists[:10]:  # Zeige nur die ersten 10
            print(f"  - {ing}")
        if len(empty_lists) > 10:
            print(f"  ... und {len(empty_lists) - 10} mehr")
    else:
        print("✅ Keine leeren Listen gefunden")
    
    # Zusammenfassung
    success = len(missing_keys) == 0 and len(empty_lists) == 0
    print(f"\n{'✅ VALIDIERUNG ERFOLGREICH' if success else '❌ VALIDIERUNG FEHLGESCHLAGEN'}")
    
    return success

# # Beispielnutzung (nach Ausführung von write_JSON_similar_ingredients_fast()):
# validate_similarity_cache("ingredient_similarity_cache_30-01-2026-13-31.json", ALL_INGREDIENTS)


Fehler beim Lesen der Datei: [Errno 2] No such file or directory: 'ingredient_similarity_cache_30-01-2026-13-31.json'


False

In [ ]:
# --- TEST mit Cache ---
cache = IngredientTokenCache(ALL_INGREDIENTS)
result = cache.check_similarity("Cucumber", thresholdHead=0.8, debug=False)
print(result)


Initialisiere Token-Cache für 877 Zutaten...
  Fortschritt: 0/877
  Fortschritt: 100/877
  Fortschritt: 200/877
  Fortschritt: 300/877
  Fortschritt: 400/877
  Fortschritt: 500/877
  Fortschritt: 600/877
  Fortschritt: 700/877
  Fortschritt: 800/877
Token-Cache bereit!
{'Cucumber': [('Cucumber', 1.0, -10), ('Persian Cucumber', 1.0, -10)]}


C:\Users\maxi9\AppData\Local\Temp\ipykernel_29376\2219725144.py:27: UserWarning: [W008] Evaluating Doc.similarity based on empty vectors.
  scoreHead = baseTokenHead.similarity(comparableTokenHead)


In [ ]:
result = cache.check_similarity("Cucumber", thresholdHead=0.8, debug=False)
print(result)

{'Cucumber': [('Cucumber', 1.0, -10), ('Persian Cucumber', 1.0, -10)]}


C:\Users\maxi9\AppData\Local\Temp\ipykernel_29376\2219725144.py:27: UserWarning: [W008] Evaluating Doc.similarity based on empty vectors.
  scoreHead = baseTokenHead.similarity(comparableTokenHead)
